---
# `Directory Loaders in Document Loaders`
---

- It is a doc-loader that lets you load Multiple Documents from a Directory folder of files
- loads files context or files as per the pattern followered by these files
- e.g folders of pdf with (.pdf) or folders of csv (.csv)


Glob Patter | What it loads
--|--
"**/*.text" | All .txt files in subfolder
"*.pdf" | All Pdf files in subfolder
"data/*.csv" | All .csv files
"**/*" | All files of any types in folder


Paramters 
- path -> Folder
- glob -> parameter defines pattern what type of file we have to add as whether if we want to load pdf, text or what
- loader-cls -> exat or actural doc loader used to load those .pdf files


Note: Enterprise Level RAG Application
- 1000s of PDF
- Per PDF more than 100 pages

load()
- this helps in eager loading as loading everything at once and return a list of Document object
- All data is loaded Directly into the Memory
- Use: The number of documents if it is small or very less 
- Use: You want everything loaded infront

lazy_load()
- It goes lazy loading and loads on demands
- Return  A generator of document objects
- Documents are not loaded all at once they are fetched one at a time as needed
- User: when you're dealing with large documents or lot of files
- You want to stream processsing eg. chunking or embeddings  

# `Detailed Notes`

# Directory Loaders in LangChain

> **Directory Loader = A component/pattern used to load multiple files from a directory and convert them into LangChain `Document` objects.**

If `TextLoader` loads **one text file**, a directory loader helps you process **many files**.

For a RAG application, this becomes extremely important because your knowledge base might contain:

```text
10 PDFs
500 PDFs
5,000 PDFs
100,000 documents
```

The basic idea is:

```text
Directory
   ↓
Find Files
   ↓
Document Loaders
   ↓
Documents
   ↓
Text Splitter
   ↓
Chunks
   ↓
Embeddings
   ↓
Vector Database
```

---

# 1. Why Do We Need Directory Loaders?

Suppose you have:

```text
knowledge_base/
├── python.txt
├── ml.txt
├── deep_learning.txt
├── nlp.txt
├── rag.txt
├── langchain.txt
├── agents.txt
└── transformers.txt
```

Without a directory loader, you might manually write:

```python
TextLoader("python.txt")
TextLoader("ml.txt")
TextLoader("deep_learning.txt")
TextLoader("nlp.txt")
...
```

That's not scalable.

Instead:

```text
knowledge_base/
       ↓
Directory Loader
       ↓
All Files
       ↓
Documents
```

---

# 2. Basic Architecture

```text
                knowledge_base/
                       │
          ┌────────────┼────────────┐
          ↓            ↓            ↓
       file1.pdf    file2.txt    file3.pdf
          │            │            │
          └────────────┼────────────┘
                       ↓
                Directory Loader
                       ↓
                   Documents
```

The important idea is:

> **Directory loading automates the process of discovering and loading multiple files.**

---

# 3. `DirectoryLoader`

LangChain provides a `DirectoryLoader` that can work with another loader to process files in a directory.

For example, if your directory contains `.txt` files:

```python
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(
    "data/",
    glob="*.txt",
    loader_cls=TextLoader
)

documents = loader.load()

print(f"Loaded {len(documents)} documents")
```

Suppose:

```text
data/
├── python.txt
├── ml.txt
├── nlp.txt
└── rag.txt
```

Then:

```text
DirectoryLoader
      ↓
   4 files
      ↓
   Documents
```

---

# 4. Understanding the Parameters

This code:

```python
loader = DirectoryLoader(
    "data/",
    glob="*.txt",
    loader_cls=TextLoader
)
```

has three important parts.

### `data/`

The directory to search.

```python
"data/"
```

### `glob`

Specifies which files should be loaded.

```python
glob="*.txt"
```

This means:

> Load all `.txt` files.

For PDFs:

```python
glob="*.pdf"
```

For Markdown:

```python
glob="*.md"
```

---

### `loader_cls`

Specifies which loader should process each file.

For TXT:

```python
loader_cls=TextLoader
```

For PDF:

```python
loader_cls=PyPDFLoader
```

So conceptually:

```text
DirectoryLoader
       ↓
Find files
       ↓
Choose appropriate loader
       ↓
Create Documents
```

---

# 5. Loading 1,000 PDF Documents

Suppose you have:

```text
documents/
├── document_001.pdf
├── document_002.pdf
├── document_003.pdf
...
├── document_1000.pdf
```

You can use:

```python
from langchain_community.document_loaders import (
    DirectoryLoader,
    PyPDFLoader
)

loader = DirectoryLoader(
    "documents/",
    glob="*.pdf",
    loader_cls=PyPDFLoader
)

documents = loader.load()

print(f"Loaded {len(documents)} documents")
```

Conceptually:

```text
1,000 PDFs
    ↓
DirectoryLoader
    ↓
PyPDFLoader
    ↓
PDF Documents
```

---

# 6. Important: Files vs Documents

There is an important distinction.

Suppose you have:

```text
1,000 PDF files
```

That does **not necessarily mean**:

```text
1,000 LangChain Documents
```

Why?

Because one PDF can contain many pages.

For example:

```text
document_001.pdf
    ↓
100 pages
    ↓
100 Document objects
```

So:

```text
1,000 PDF files
        ↓
possibly tens of thousands
of Document objects
```

The exact behavior depends on the loader and configuration.

### Interview Point

> **A file and a LangChain `Document` are not necessarily one-to-one. A single PDF can produce multiple `Document` objects, for example one per page.**

---

# 7. Directory Structure for a Real RAG System

A real knowledge base might look like:

```text
knowledge_base/
│
├── python/
│   ├── basics.pdf
│   ├── oop.pdf
│   └── advanced.pdf
│
├── machine-learning/
│   ├── regression.pdf
│   ├── classification.pdf
│   └── clustering.pdf
│
├── deep-learning/
│   ├── cnn.pdf
│   ├── rnn.pdf
│   └── transformers.pdf
│
└── genai/
    ├── llm.pdf
    ├── rag.pdf
    ├── agents.pdf
    └── langchain.pdf
```

You can recursively search directories.

Conceptually:

```text
knowledge_base/
       ↓
Recursive Directory Search
       ↓
All PDFs
       ↓
PDF Loader
       ↓
Documents
```

---

# 8. Recursive Loading

If your documents are inside subdirectories, you generally want recursive discovery.

Conceptually:

```text
knowledge_base/
├── python/
│   ├── basics.pdf
│   └── oop.pdf
│
├── ml/
│   ├── regression.pdf
│   └── classification.pdf
│
└── genai/
    ├── rag.pdf
    └── agents.pdf
```

The loader should discover:

```text
python/basics.pdf
python/oop.pdf
ml/regression.pdf
ml/classification.pdf
genai/rag.pdf
genai/agents.pdf
```

Depending on your LangChain version, the `recursive` option can be used with `DirectoryLoader`:

```python
loader = DirectoryLoader(
    "knowledge_base/",
    glob="**/*.pdf",
    loader_cls=PyPDFLoader,
    recursive=True
)
```

The key idea is:

```text
*      → current directory pattern
**/*   → recursive directory pattern
```

---

# 9. Loading Different File Types

Real knowledge bases rarely contain only PDFs.

You might have:

```text
knowledge_base/
├── python.pdf
├── notes.txt
├── architecture.md
├── data.csv
└── documentation.pdf
```

The problem is:

> Different file types require different loaders.

For example:

```text
PDF  → PyPDFLoader
TXT  → TextLoader
CSV  → CSVLoader
MD   → Markdown loader
DOCX → Word/document loader
```

So a single `loader_cls` isn't always enough for a mixed directory.

---

# 10. Solution: File-Type-Based Loading

A production-friendly approach is to map file extensions to loaders.

Conceptually:

```python
LOADERS = {
    ".pdf": PyPDFLoader,
    ".txt": TextLoader,
    ".md": MarkdownLoader
}
```

Then:

```text
File
 ↓
Check extension
 ↓
Choose loader
 ↓
Load
 ↓
Document
```

This gives you:

```text
PDF → PDF Loader
TXT → Text Loader
MD  → Markdown Loader
```

---

# 11. Why Loading 1,000 Documents Is Different From Loading 10

For:

```text
10 documents
```

you can often do:

```python
documents = loader.load()
```

without worrying much.

But for:

```text
10,000 documents
```

you need to think about:

* Memory
* Processing time
* API rate limits
* Embedding cost
* Failures
* Duplicate documents
* Incremental updates
* Parallel processing
* Batch processing
* Checkpointing

This is where you move from a **tutorial implementation** toward a **production ingestion pipeline**.

---

# 12. The Naive Approach

Imagine:

```text
100,000 PDFs
```

You do:

```python
documents = loader.load()
```

Then:

```text
100,000 PDFs
       ↓
Load EVERYTHING
       ↓
Store EVERYTHING in RAM
       ↓
Split EVERYTHING
       ↓
Embed EVERYTHING
       ↓
Store EVERYTHING
```

Potential problems:

```text
High Memory Usage
Long Processing Time
Failure → Restart From Beginning
```

This is not ideal for a large production system.

---

# 13. Better Approach: Batch Processing

Instead of:

```text
10,000 files
 ↓
Process all at once
```

process batches:

```text
10,000 files
     ↓
┌──────────────┐
│ Batch 1      │ 100 files
├──────────────┤
│ Batch 2      │ 100 files
├──────────────┤
│ Batch 3      │ 100 files
├──────────────┤
│ ...          │
└──────────────┘
```

Pipeline:

```text
Batch
 ↓
Load
 ↓
Split
 ↓
Embed
 ↓
Store
 ↓
Release Memory
 ↓
Next Batch
```

---

# 14. Why Batch Processing Is Better

Suppose you have:

```text
50,000 PDFs
```

Instead of:

```text
50,000 PDFs → RAM
```

you can process:

```text
500 PDFs
 ↓
Process
 ↓
Store
 ↓
Clear
 ↓
Next 500
```

Advantages:

* Lower memory usage
* Easier failure recovery
* Better control over API calls
* Easier monitoring
* Easier scaling

---

# 15. Use `lazy_load()` for Large Data

For large collections, lazy loading can be useful.

Conceptually:

```python
documents = loader.lazy_load()

for document in documents:
    process(document)
```

Instead of immediately materializing all documents:

```text
load()
 ↓
Everything immediately
```

lazy loading works more like:

```text
lazy_load()
 ↓
Document 1
 ↓
Document 2
 ↓
Document 3
 ↓
...
```

### Important

Lazy loading helps control memory, but it doesn't automatically make your entire ingestion pipeline scalable. You still need to think about batching, embeddings, retries, and vector-store writes.

---

# 16. Large-Scale RAG Architecture

For thousands or millions of documents, think beyond `DirectoryLoader`.

A better architecture is:

```text
                  DOCUMENT STORAGE
                         ↓
              ┌────────────────────┐
              │ PDFs / TXT / DOCX  │
              └──────────┬─────────┘
                         ↓
                  File Discovery
                         ↓
                   Batch Queue
                         ↓
                  Document Loader
                         ↓
                  Text Extraction
                         ↓
                  Text Splitter
                         ↓
                    Chunks
                         ↓
                  Embedding Batch
                         ↓
                  Vector Database
                         ↓
                   Ingestion Done
```

Then:

```text
                    USER QUERY
                         ↓
                     Retriever
                         ↓
                 Vector Database
                         ↓
                  Relevant Chunks
                         ↓
                        LLM
                         ↓
                      Answer
```

---

# 17. A Production Mindset

When someone asks:

> "How do you load 100,000 documents into a RAG system?"

Don't just answer:

```python
DirectoryLoader(...).load()
```

That's only the **file-discovery/loading part**.

Think:

```text
1. Discover files
2. Validate files
3. Load files
4. Extract content
5. Preserve metadata
6. Split into chunks
7. Deduplicate
8. Generate embeddings in batches
9. Store vectors
10. Track ingestion status
11. Retry failures
12. Support incremental updates
```

That's a much stronger interview answer.

---

# 18. Metadata Becomes Extremely Important

Suppose you have:

```text
10,000 documents
```

You should preserve metadata such as:

```text
source
file_name
file_path
file_type
document_id
page_number
category
created_at
updated_at
```

Example:

```python
{
    "source": "knowledge_base/genai/rag.pdf",
    "file_name": "rag.pdf",
    "category": "genai",
    "page": 12
}
```

Then you can perform filtering.

For example:

> Search only documents from the `genai` category.

---

# 19. Deduplication

Large document collections often contain duplicates.

Example:

```text
report.pdf
report_copy.pdf
report_final.pdf
report_final_v2.pdf
```

They may contain the same content.

If you embed all of them:

```text
Duplicate Content
      ↓
Duplicate Vectors
      ↓
Poor Retrieval
      ↓
Higher Cost
```

A production pipeline should consider deduplication using:

* File hashes
* Content hashes
* Document IDs
* Version IDs

---

# 20. Incremental Ingestion

This is one of the **most important production concepts**.

Suppose you initially have:

```text
1,000 documents
```

You ingest all of them.

Tomorrow:

```text
20 new documents
```

You don't want to reprocess:

```text
1,000 old documents + 20 new
```

Instead:

```text
Existing 1,000
       ↓
Already indexed

New 20
       ↓
Process only new documents
```

This is called **incremental ingestion**.

---

# 21. Detecting New or Updated Documents

You can track:

```text
document_id
file_hash
last_modified
version
```

Example:

```text
document.pdf
    ↓
SHA-256 hash
    ↓
abc123...
```

If the file hasn't changed:

```text
Same hash
 ↓
Skip
```

If changed:

```text
Different hash
 ↓
Reprocess
```

---

# 22. Handling Failed Documents

Imagine you have:

```text
10,000 documents
```

and:

```text
document_582.pdf
```

is corrupted.

You don't want:

```text
10,000 files
     ↓
582 succeeds
     ↓
583 fails
     ↓
ENTIRE PIPELINE STOPS
```

Instead:

```text
Document 582
     ↓
ERROR
     ↓
Log Error
     ↓
Continue
     ↓
Document 583
```

Maintain an ingestion status:

```text
document_001 → SUCCESS
document_002 → SUCCESS
document_003 → FAILED
document_004 → SUCCESS
```

Then retry only failures.

---

# 23. Batch Embeddings

Embedding thousands of documents one-by-one can be inefficient.

Bad:

```text
Chunk 1 → API call
Chunk 2 → API call
Chunk 3 → API call
...
```

Better:

```text
Chunk 1 ─┐
Chunk 2 ─┤
Chunk 3 ─┤
Chunk 4 ─┤
         ↓
   Batch Embedding
         ↓
      Vectors
```

This can improve throughput and help manage API limits.

---

# 24. Parallel Processing

Suppose:

```text
10,000 PDFs
```

Processing one at a time may be slow.

You can potentially parallelize independent work:

```text
                10,000 PDFs
                     ↓
            ┌────────┼────────┐
            ↓        ↓        ↓
         Worker 1 Worker 2 Worker 3
            ↓        ↓        ↓
          Load     Load     Load
            ↓        ↓        ↓
          Split    Split    Split
            ↓        ↓        ↓
         Embeddings Embeddings
            └────────┼────────┘
                     ↓
               Vector Store
```

But parallelism must be controlled.

Too much concurrency can cause:

* API rate limits
* Memory pressure
* Database overload
* CPU exhaustion

---

# 25. Recommended Large-Scale Architecture

For a serious production system:

```text
                    File Storage
                  S3 / Cloud Storage
                         ↓
                  File Discovery
                         ↓
                   Message Queue
                         ↓
              ┌──────────┴──────────┐
              ↓                     ↓
         Worker 1               Worker 2
              ↓                     ↓
         PDF Loader             PDF Loader
              ↓                     ↓
         Text Splitter          Text Splitter
              ↓                     ↓
         Embedding              Embedding
              ↓                     ↓
              └──────────┬──────────┘
                         ↓
                  Vector Database
```

For example, you might use:

```text
Storage → S3
Queue → Kafka / SQS
Workers → Python services
Vector DB → Qdrant / Pinecone / Weaviate
```

The exact technology depends on the project.

---

# 26. Mini Project: AI Knowledge Base

Let's turn this into a practical project.

## Project Goal

Build:

> **Enterprise Knowledge Base RAG**

The system accepts thousands of company documents.

```text
Company Documents
       ↓
      RAG
       ↓
AI Knowledge Assistant
```

Users can ask:

```text
"What is our leave policy?"

"What is the reimbursement process?"

"Explain the security policy."

"Which document discusses remote work?"
```

---

# 27. Project Structure

```text
enterprise-rag/
│
├── data/
│   ├── hr/
│   │   ├── leave.pdf
│   │   ├── benefits.pdf
│   │   └── reimbursement.pdf
│   │
│   ├── engineering/
│   │   ├── architecture.pdf
│   │   └── coding-guidelines.pdf
│   │
│   └── security/
│       ├── security-policy.pdf
│       └── access-control.pdf
│
├── ingestion/
│   ├── discover.py
│   ├── loader.py
│   ├── splitter.py
│   ├── embedder.py
│   └── pipeline.py
│
├── retrieval/
│   └── retriever.py
│
├── app/
│   └── chatbot.py
│
└── requirements.txt
```

---

# 28. Step 1 — Discover Documents

```python
from pathlib import Path

files = list(Path("data").rglob("*.pdf"))

print(f"Found {len(files)} PDFs")

for file in files[:10]:
    print(file)
```

If there are 5,000 PDFs:

```text
Found 5000 PDFs
```

Now you've separated:

> **File discovery from document loading.**

That's a good production design.

---

# 29. Step 2 — Load One File at a Time

Instead of immediately loading everything:

```python
from langchain_community.document_loaders import PyPDFLoader

def load_pdf(file_path):
    loader = PyPDFLoader(str(file_path))
    return loader.load()
```

Then:

```python
documents = load_pdf(files[0])
```

Now you can control exactly how each file is processed.

---

# 30. Step 3 — Process in Batches

Conceptually:

```python
BATCH_SIZE = 50

for i in range(0, len(files), BATCH_SIZE):

    batch = files[i:i + BATCH_SIZE]

    for file_path in batch:

        documents = load_pdf(file_path)

        # Split
        # Embed
        # Store

    print(f"Processed batch {i // BATCH_SIZE + 1}")
```

Architecture:

```text
5,000 PDFs
    ↓
Batch 1 → 50 PDFs
    ↓
Process
    ↓
Store
    ↓
Batch 2 → 50 PDFs
    ↓
Process
    ↓
Store
    ↓
...
```

---

# 31. Step 4 — Add Metadata

When processing each file, attach useful metadata.

```python
for document in documents:

    document.metadata["file_name"] = file_path.name
    document.metadata["file_path"] = str(file_path)
    document.metadata["category"] = file_path.parent.name
```

Now:

```python
{
    "source": "...",
    "page": 5,
    "file_name": "leave.pdf",
    "category": "hr"
}
```

This becomes extremely useful for retrieval.

---

# 32. Step 5 — Split

```python
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(documents)
```

Now:

```text
PDF
 ↓
Documents
 ↓
Chunks
```

---

# 33. Step 6 — Embed and Store

Conceptually:

```text
Chunks
 ↓
Embedding Model
 ↓
Vectors
 ↓
Vector Database
```

Do this in batches rather than creating one embedding request per chunk whenever your embedding provider supports batching.

---

# 34. Step 7 — Track Ingestion

Create a simple tracking system:

```text
document_id
status
hash
processed_at
error
```

Example:

```text
leave.pdf
SUCCESS
hash=abc123
processed_at=2026-08-10
```

If processing fails:

```text
broken.pdf
FAILED
error="Unable to parse PDF"
```

---

# 35. Final Production Pipeline

For thousands of documents, think:

```text
                  DOCUMENT DIRECTORY
                         ↓
                  File Discovery
                         ↓
                 Filter / Validate
                         ↓
                   Batch Files
                         ↓
                Document Loader
                         ↓
                   Documents
                         ↓
                Metadata Enrichment
                         ↓
                  Text Splitting
                         ↓
                    Deduplication
                         ↓
                 Batch Embeddings
                         ↓
                  Vector Database
                         ↓
                Ingestion Tracking
```

Then:

```text
                      USER
                       ↓
                     Query
                       ↓
                   Retriever
                       ↓
                Vector Database
                       ↓
               Relevant Chunks
                       ↓
                      LLM
                       ↓
                    Answer
                       ↓
              Source + Page Citation
```

---

# 36. `DirectoryLoader` vs Production Ingestion Pipeline

This distinction is **very important for interviews**.

### Small project

You can use:

```python
DirectoryLoader(...).load()
```

Simple and convenient.

### Thousands/millions of documents

You should think about:

```text
File Discovery
+
Batch Processing
+
Lazy Loading
+
Parallel Workers
+
Retries
+
Deduplication
+
Incremental Ingestion
+
Metadata
+
Monitoring
+
Checkpointing
```

So don't say:

> "I would use DirectoryLoader to load 1 million documents into memory."

Instead say:

> **"I can use DirectoryLoader for simple directory-based ingestion, but for thousands or millions of documents I would build a batch/streaming ingestion pipeline that discovers files, processes them incrementally, embeds in batches, tracks failures, and writes to the vector database progressively."**

That's a much stronger answer.

---

# 37. Common Interview Questions

## Beginner

### Q1. What is `DirectoryLoader`?

**Answer:**

`DirectoryLoader` helps load multiple files from a directory using a specified document loader.

---

### Q2. Why use `DirectoryLoader` instead of `TextLoader`?

**Answer:**

`TextLoader` is typically used for an individual text file, while `DirectoryLoader` can discover multiple files and use a loader to process them.

---

### Q3. How do you load all PDF files?

```python
loader = DirectoryLoader(
    "data/",
    glob="*.pdf",
    loader_cls=PyPDFLoader
)

documents = loader.load()
```

---

## Intermediate

### Q4. How do you load PDFs from nested directories?

Use recursive file discovery/patterns.

Conceptually:

```python
glob="**/*.pdf"
```

This can discover PDFs in subdirectories.

---

### Q5. Does 1,000 files mean 1,000 Documents?

**Answer:**

Not necessarily. One file can produce multiple `Document` objects. For example, a PDF loader may produce documents representing individual pages.

---

### Q6. How would you process 100,000 documents?

**Answer:**

I would avoid loading everything into memory. I'd use file discovery, batching/lazy processing, parallel workers where appropriate, batch embeddings, incremental indexing, retries, metadata, deduplication, and ingestion tracking.

---

# 38. Scenario-Based Interview Questions

### Q7. You have 50,000 PDFs. `loader.load()` causes memory problems. What would you do?

**Answer:**

I would avoid materializing all documents at once. I'd process files incrementally or in batches, use lazy loading where appropriate, split and embed each batch, write results to the vector database, and release memory before processing the next batch.

---

### Q8. Your ingestion process fails after processing 8,000 of 10,000 files. What would you do?

**Answer:**

I would maintain ingestion status and checkpoints for each document. Then I could identify failed/unprocessed files and resume from there instead of reprocessing all 10,000 documents.

---

### Q9. You receive 100 new documents every day. Would you re-index everything?

**Answer:**

No. I would use incremental ingestion. Newly added or modified documents would be detected and processed, while unchanged documents would be skipped.

---

### Q10. Your directory contains PDF, TXT, DOCX, and CSV files. Can you use one loader?

**Answer:**

Usually, different file types require different loaders. I'd use file-extension-based routing:

```text
.pdf  → PDF Loader
.txt  → Text Loader
.docx → DOCX Loader
.csv  → CSV Loader
```

---

# 39. 30-Second Revision

> **Directory Loader helps load multiple files from a directory using appropriate document loaders.**

```text
Directory
   ↓
File Discovery
   ↓
DirectoryLoader
   ↓
Document Loader
   ↓
Documents
```

For RAG:

```text
Files
 ↓
Load
 ↓
Split
 ↓
Embed
 ↓
Vector DB
 ↓
Retrieve
 ↓
LLM
```

### For Thousands of Documents

Don't think only:

```text
DirectoryLoader → load()
```

Think:

```text
Discover
 ↓
Batch
 ↓
Load
 ↓
Split
 ↓
Embed
 ↓
Store
 ↓
Track
 ↓
Next Batch
```

---

# 40. 2-Minute Revision

## Directory Loader

Used to load multiple files from a directory.

### Basic Example

```python
from langchain_community.document_loaders import (
    DirectoryLoader,
    TextLoader
)

loader = DirectoryLoader(
    "data/",
    glob="*.txt",
    loader_cls=TextLoader
)

documents = loader.load()
```

### PDF Example

```python
from langchain_community.document_loaders import (
    DirectoryLoader,
    PyPDFLoader
)

loader = DirectoryLoader(
    "data/",
    glob="*.pdf",
    loader_cls=PyPDFLoader
)

documents = loader.load()
```

### Recursive Directory

```text
knowledge_base/
├── python/
│   └── basics.pdf
├── ml/
│   └── regression.pdf
└── genai/
    └── rag.pdf
```

Use recursive file discovery so nested PDFs are found.

### Large-Scale Ingestion

```text
                 Thousands of Files
                         ↓
                  File Discovery
                         ↓
                   Validation
                         ↓
                    Batching
                         ↓
                 Document Loader
                         ↓
                  Text Splitting
                         ↓
                  Deduplication
                         ↓
                Batch Embeddings
                         ↓
                  Vector Database
                         ↓
               Ingestion Tracking
```

### Production Checklist

```text
✓ Batch processing
✓ Lazy/incremental loading
✓ Metadata
✓ Deduplication
✓ Batch embeddings
✓ Retry failed documents
✓ Checkpointing
✓ Incremental updates
✓ Monitoring
✓ Error logging
```

### Most Important Interview Concept

> **`DirectoryLoader` is convenient for loading multiple files, but for thousands or millions of documents, a production RAG system should use incremental/batch ingestion with controlled concurrency, metadata, retries, deduplication, and checkpointing rather than loading the entire corpus into memory at once.**
